# Stage 7: Feature Engineering & Preprocessing Pipeline
## Student Performance Analyzer (HCL Career Shaper Program)

### Objectives
1. Implement domain-informed pre-outcome feature transformations via `src/features.py`.
2. Construct and verify features:
   - `parent_edu_avg`
   - `total_alc`
   - `support_index`
   - `absences_capped`
   - `attendance_rate_est`
3. **Enforce Target Leakage Quarantining:** Strictly eliminate $G1, G2, G3$ from feature matrix $X$.
4. Implement stratified 80/20 train/test splitting (`random_state=42`).
5. Build and validate Scikit-learn `ColumnTransformer` with `StandardScaler` and `OneHotEncoder`.
6. Ensure preprocessing transformers fit ONLY on training data.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))
from src.cleaning import load_raw_data, clean_student_data
from src.features import (
    ALL_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    LEAKAGE_COLUMNS,
    build_preprocessor,
    engineer_features,
    get_feature_target_split,
    split_data,
)


### 1. Ingest Cleaned Data & Apply Pre-Outcome Feature Engineering

In [ ]:
raw = load_raw_data("../data/raw/student-mat.csv")
clean_df = clean_student_data(raw)
feat_df = engineer_features(clean_df)

print(f"Features DataFrame dimensions: {feat_df.shape}")
feat_df[["Medu", "Fedu", "parent_edu_avg", "Dalc", "Walc", "total_alc", "support_index", "absences", "absences_capped", "attendance_rate_est"]].head()


### 2. Isolate Predictor Matrix $X$ and Target $y$ (Zero Leakage Check)

In [ ]:
X, y = get_feature_target_split(feat_df, target_col="passed")
print(f"Feature matrix X shape: {X.shape}")
print(f"Target series y shape: {y.shape}")

# Strict Verification: Guarantee NO leakage columns in X
for col in LEAKAGE_COLUMNS:
    assert col not in X.columns, f"LEAKAGE ALERT: {col} is present in X!"
print("Zero Target Leakage Verification: ALL period grades (G1, G2, G3) successfully quarantined!")


### 3. Stratified Train / Test Partitioning (80/20)
We reserve an untouched holdout test set (20% of data = 79 records). The split is stratified by $y$ to preserve the 67% pass / 33% fail balance.


In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42, stratify=True)
print(f"Training set: {X_train.shape[0]} rows (Pass rate: {y_train.mean():.4f})")
print(f"Test set:     {X_test.shape[0]} rows (Pass rate: {y_test.mean():.4f})")


### 4. Build and Fit Scikit-Learn Preprocessing Pipeline
We employ `ColumnTransformer` to:
- Standardize all 18 numerical features using `StandardScaler`.
- One-hot encode all 17 categorical features using `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.

*Crucial rule: Preprocessor is fitted strictly on `X_train` to prevent data snooping.*


In [ ]:
preprocessor = build_preprocessor()
preprocessor.fit(X_train)

X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Transformed X_train shape: {X_train_trans.shape}")
print(f"Transformed X_test shape:  {X_test_trans.shape}")
print(f"Total encoded features: {len(feature_names)}")
print("\nFirst 15 transformed feature names:\n", list(feature_names[:15]))


### Key Findings (Stage 7)
1. **Pre-Outcome Integrity:** Five engineered features (`parent_edu_avg`, `total_alc`, `support_index`, `absences_capped`, `attendance_rate_est`) enrich model representation without requiring exam scores.
2. **Quarantine Verified:** $G1, G2, G3$ are completely omitted from the predictor matrix.
3. **Stratification Maintained:** Both train (316 rows) and test (79 rows) sets have an identical 67.09% pass proportion.
4. **Leak-Free Transformation:** Preprocessor scales and encodes strictly using training set parameters, expanding the raw features into 61 standardized dimensions.
